# **Projeto Final — Aprendizado de Máquina I (2026.2) · Parte II: aplicação**
### Grupo: *Machine Learning Name in progress*

- Tipo de problema: **Regressão**
- Conjunto de dados: [**SRAG — Sivep-Gripe**](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026)
- Objetivo principal: **estimar as notificações semanais de SRAG com VSR em bebês menores de 6 meses em um cenário de referência sem a campanha de vacinação materna**


- **Equipe responsável**
    - Ryan Kevin
    - Gabriel Falcão
    - Marcelo Lopes
- **Revisão** 
    - Lucas Vieira
    - Mateus Almeida

## 0. Configuração e reprodutibilidade

- A base com o recorte de até 4 anos completos está em `data/vsr.csv`.
- A célula de leitura usa pandas. A equipe deverá registrar aqui a versão da biblioteca do grupo e a semente dos modelos que precisarem dela.
- Execute o notebook a partir da raiz do repositório.

## 1. Problema e dados

**Pergunta de regressão:** se a relação observada antes da campanha de vacinação materna contra o VSR se mantivesse, quantas notificações semanais de SRAG com VSR esperaríamos em bebês de **0 a menos de 6 meses** em 2026?

- **Alvo:** contagem de notificações de SRAG com VSR no grupo de 0 a <6 meses.
- **Entradas:** contagens de SRAG com VSR na mesma semana em 6 a <12, 12 a <24 e 24 a <48 meses.
- **Unidade de análise:** uma semana nacional, agrupada pela data dos primeiros sintomas.
- **Referência histórica:** usar dados anteriores à campanha do SUS, iniciada em 04/12/2025. Reservar um período anterior à campanha para o teste. O arquivo reúne os extratos de 2023 a 2026. Usar a data dos primeiros sintomas para separar o histórico anterior à campanha das semanas do cenário em 2026.

O cenário supõe que a relação histórica entre as faixas etárias se manteria. As faixas mais velhas servem como referência para a circulação do vírus. Como algumas crianças menores de 24 meses são elegíveis ao nirsevimabe, testamos a sensibilidade usando apenas 24 a <48 meses. A diferença entre o observado e o estimado depende dessa hipótese e pode refletir outras mudanças, como testagem e notificação. 
[Início da campanha](https://www.gov.br/saude/pt-br/assuntos/noticias/2025/dezembro/gestantes-ja-podem-se-vacinar-contra-a-bronquiolite)
[Nota técnica sobre nirsevimabe](https://www.gov.br/saude/pt-br/centrais-de-conteudo/publicacoes/notas-tecnicas/2026/nota-tecnica-conjunta-no-87-2026-svsa-saps-saes-sesai-ms.pdf/%40%40download/file)

### 1.1 Carregamento dos dados

O arquivo contém as notificações dos extratos de 2023 a 2026 para crianças de **até 4 anos completos**, ou seja, desde o nascimento até antes do quinto aniversário. A base reúne 514.225 notificações e mantém os valores das cinco colunas usadas na análise. O recorte de idade foi aplicado antes da seleção de colunas.

Como a idade é registrada: `NU_IDADE_N` é um número e `TP_IDADE` indica a unidade. Por exemplo, idade 4 com unidade 2 significa **4 meses**, enquanto idade 4 com unidade 3 significa **4 anos completos**. A idade é registrada na data dos primeiros sintomas.

| `TP_IDADE` | Unidade | Valores de `NU_IDADE_N` incluídos |
|---|---|---|
| 1 | Dias | 0 a 31 |
| 2 | Meses | 0 a 59 |
| 3 | Anos completos | 0 a 4 |

Registros com idade ausente ou unidade inválida ficam fora do recorte. Uma criança registrada com 4 anos entra na base, mas fica fora da faixa explicativa de **24 a menos de 48 meses**. A equipe usará as faixas de 0 a <6, 6 a <12, 12 a <24 e 24 a <48 meses ao construir o painel.


In [ ]:
import pandas as pd

DATA_PATH = "data/vsr.csv"
df = pd.read_csv(DATA_PATH, sep=";", low_memory=False)
df.shape

### 1.2 Campos de referência e origem

| Coluna | Uso no pipeline |
|---|---|
| `DT_SIN_PRI` | Data dos primeiros sintomas. Usar para agrupar as notificações por semana. |
| `TP_IDADE`, `NU_IDADE_N` | Unidade e valor da idade. Usar em conjunto para construir as faixas etárias. |
| `PCR_VSR`, `AN_VSR` | Indicadores de detecção de VSR. Ao montar o alvo e as entradas, considerar positivo quando pelo menos um deles for 1 e contar a notificação uma vez. |

Extratos do Sivep-Gripe usados no CSV: [2023](https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2023/INFLUD23-23-03-2026.parquet) e [2024](https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2024/INFLUD24-23-03-2026.parquet), de 23/03/2026; [2025](https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2025/INFLUD25-02-11-2026.parquet) e [2026](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026/resource/74091efc-3f75-42e8-a6fa-6b79a8d30582), de 02/11/2026. Foram mantidos apenas a data dos sintomas, o valor e a unidade da idade e os dois indicadores de detecção de VSR. Os extratos históricos em Parquet foram convertidos para CSV. [Dicionário de dados](https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/dicionario-de-dados-2019-a-2025.pdf)

## 2. Análise exploratória (EDA)

### 2.1 Dimensões, tipos e primeiras linhas

Conferir datas, códigos de idade e valores dos indicadores de VSR.

### 2.2 Valores ausentes

Verificar datas e indicadores ausentes. Campo de exame vazio não equivale a um teste negativo.

### 2.3 Distribuição da variável-alvo

Explorar as contagens semanais de 0 a <6 meses no período de desenvolvimento.

### 2.4 Distribuição das variáveis explicativas

Explorar as três faixas e identificar sazonalidade.

### 2.5 Relação entre variáveis

Investigar correlação e colinearidade entre as contagens semanais.

### 2.6 Conclusões da EDA

Registrar as decisões e limitações encontradas.

## 3. Divisão treino/teste

### 3.1 Unidade de observação

Conferir que o painel agregado tem uma linha por semana. Linhas iguais nas cinco colunas podem representar notificações distintas. A base não contém identificadores para deduplicação individual.

### 3.2 Divisão

Separar períodos em ordem cronológica. Reservar um trecho pré-campanha de 2025 para o teste e deixar 2026 para o cenário.

### 3.3 Conferência

Verificar datas, semanas completas e quantidade de semanas por conjunto.

## 4. Pré-processamento

### 4.1 Construção do painel

Filtrar VSR detectado, definir faixas sem sobreposição e agregar pela semana de início dos sintomas.

### 4.2 Valores ausentes

Definir como tratar datas inválidas e distinguir semana sem notificações de semana sem cobertura.

### 4.3 Escala das variáveis numéricas

Se padronizar, aprender os parâmetros apenas nas semanas de treino de cada divisão.

### 4.4 Matrizes finais

Montar as três colunas explicativas e o alvo, sem incorporar dados do futuro.

## 5. Modelos candidatos

### 5.1 Baseline

Definir uma previsão simples para servir de referência.

### 5.2 Modelos de regressão

Escolher regressões disponíveis na biblioteca do grupo e justificar os hiperparâmetros.

### 5.3 Rede neural

Completar a comparação com o regressor neural previsto no roteiro.

## 6. Validação e seleção de hiperparâmetros

### 6.1 Função de validação

Usar blocos temporais. Em cada bloco, treinar apenas com as semanas anteriores.

### 6.2 Busca de hiperparâmetros

Definir a grade e uma métrica principal, como MAE em notificações por semana.

## 7. Comparação de modelos

### 7.1 Escolha do modelo final

Comparar as métricas da validação e justificar a escolha antes de consultar o teste.

## 8. Avaliação final no conjunto de teste

### 8.1 Treino final

Ajustar o modelo escolhido nas semanas de desenvolvimento.

### 8.2 Métricas e gráficos

Medir o erro no teste pré-campanha e comparar observado e previsto ao longo do tempo.

### 8.3 Cenário de referência para 2026

Após o teste, ajustar com o histórico pré-campanha e estimar as semanas de 2026 a partir das faixas de referência. Justificar o recorte das semanas recentes.

## 9. Conclusão

### 9.1 Interpretação

Resumir o desempenho e discutir a hipótese de estabilidade, o atraso de notificação e o alcance da comparação.